# INF201 Deliverable 3

This notebook presents the six classes required for the deliverable, checks their behavior, and compares complete battles with and without weapons. The hero always attacks first, so battle order can affect the result.

In [1]:
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon
from battle import Battle

## Character

Character stores name, hp_max, damage and an optional weapon which can be used to attack the enemy with a random dice roll. The protected _hp attribute is exposed through read-only hp and is_alive properties. take_damage(amount) and heal(amount) clamp health to the allowed range. attack(target) calls the target's take_damage method, allowing armor to work through polymorphism. introduce(), __str__() and __repr__() describe the character in print statements. Negative maximum HP is rejected by the code.

In [2]:
character = Character("Jens", hp_max=40, damage=8)
character.introduce()
print(str(character))
print(repr(character))
character.take_damage(100)
assert character.hp == 0 and not character.is_alive
character.heal(100)
assert character.hp == 40 and character.is_alive
for attribute in ("hp", "is_alive"):
    try:
        setattr(character, attribute, 0)
    except AttributeError:
        pass
    else:
        raise AssertionError(f"{attribute} must be read-only")
try:
    Character("Invalid", hp_max=-1)
except ValueError:
    print("Negative maximum HP rejected.")
print("Health boundaries and read-only properties passed.")

Hi, my name is Jens!
Character: Jens with 40 HP and 8 damage
<Character name=Jens hp=40 damage=8>
Negative maximum HP rejected.
Health boundaries and read-only properties passed.


## Dice

Dice has a sides attribute with a default of 6,and a minimum of 2. roll() samples an integer from 1 through sides. __call__() lets d20() roll the same way as d20.roll(). __repr__() shows the configured sides.

In [3]:
d6 = Dice()
d20 = Dice(20)
print(repr(d6), "d20 roll:", d20.roll(), "d20 call:", d20())
assert d6.sides == 6 and repr(d6) == "Dice(sides=6)"
assert all(1 <= d20() <= 20 for _ in range(100))
try:
    Dice(1)
except ValueError:
    print("Dice with fewer than two sides rejected.")
else:
    raise AssertionError("Invalid dice accepted")

Dice(sides=6) d20 roll: 7 d20 call: 4
Dice with fewer than two sides rejected.


## Weapon

Weapon stores a name and damage_dice, a Dice object. roll_damage() delegates to that die. Character.attack() uses the equipped weapon instead of base damage. This composition lets weapons choose different dice without changing Character.

In [ ]:
sword = Weapon("Terrablade", Dice(8))

print(sword.name)
print(sword.damage_dice)
print(sword.roll_damage())

assert sword.name == "Terrablade"
assert sword.damage_dice.sides == 8

Terrablade
Dice(sides=8)
3


## Warrior

Warrior inherits Character attributes and methods and adds armor (default 5). Its take_damage() override subtracts armor from incoming damage, with a minimum of zero, before using the parent method.

In [5]:
warrior = Warrior("Håland", armor=5)
warrior.take_damage(12)
assert warrior.hp == 93

warrior.take_damage(3)
assert warrior.hp == 93

## Mage

Mage inherits Character and adds mana and mana_max as a comabt resource. With at least 10 mana, attack() spends 10 and deals twice the base damage, ignoring weapons. Otherwise it uses Character.attack(), including an equipped weapon, and restores 5 mana up to mana_max. Initial mana is kept between zero and mana_max.

In [6]:
mage = Mage("Solas", mana=10, damage=10)
enemy = Character("Enemy")

mage.attack(enemy)
assert mage.mana == 0
assert enemy.hp == 80

mage.attack(enemy)
assert mage.mana == 5
assert enemy.hp == 70

# Test mana boundaries
mage2 = Mage("Saruman", mana=50, mana_max=30)
assert mage2.mana == 30

mage3 = Mage("Hoodini", mana=-10)
assert mage3.mana == 0

## Battle

Battle stores hero and enemy. play_round() lets the hero attack first and permits a counterattack only if the enemy survives. simulate_battle(max_rounds=1000) repeats rounds until a character dies and prints the winner. It raises RuntimeError at the round limit to prevent endless combat, without treating one unlucky dice roll as a stalemate, this can occur in a fight between two warriors. A completed battle cannot attack again.

In [7]:
hero = Character("Link", hp_max=30, damage=10)
enemy = Character("Ganondorf", hp_max=20, damage=5)

battle = Battle(hero, enemy)
battle.simulate_battle()

assert hero.hp == 25
assert enemy.hp == 0
assert hero.is_alive is True
assert enemy.is_alive is False

Link HP: 25, Ganondorf HP: 10
Link HP: 25, Ganondorf HP: 0
Ganondorf has been defeated by Link. YIPPEE!


## Weapon behavior during Mage attacks

A fixed-damage test die isolates the rules from randomness. The first attack ignores the weapon and consumes mana; the next uses the weapon and restores mana.

In [8]:
class FixedDice(Dice):
    def roll(self):
        return 7

staff = Weapon("Last Prism", FixedDice(8))
mage = Mage("Arceus", mana=10, damage=10, weapon=staff)
target = Character("The Jonkler")
mage.attack(target)
assert target.hp == 80 and mage.mana == 0
mage.attack(target)
assert target.hp == 73 and mage.mana == 5
print("Checks passed.")

Checks passed.


## Warrior versus Mage without weapons

These characters rely on base damage and spells.

In [9]:
unarmed_warrior = Warrior("Jitterbug", hp_max=60, damage=12)
unarmed_mage = Mage("Borat", hp_max=60, damage=10)
Battle(unarmed_warrior, unarmed_mage).simulate_battle()
assert unarmed_warrior.is_alive != unarmed_mage.is_alive
print("Remaining mana:", unarmed_mage.mana)

Jitterbug HP: 45, Borat HP: 48
Jitterbug HP: 30, Borat HP: 36
Jitterbug HP: 15, Borat HP: 24
Jitterbug HP: 10, Borat HP: 12
Jitterbug HP: 10, Borat HP: 0
Borat has been defeated by Jitterbug. YIPPEE!
Remaining mana: 5


## Warrior versus Mage with weapons

Fresh characters use the same health and base damage as above. Weapon damage replaces base damage, so equipping a weapon can lower damage as well as make it variable. Mage spells still ignore the staff.

In [10]:
armed_warrior = Warrior("Neo with a gun", hp_max=60, damage=12,
                        weapon=Weapon("Glock", Dice(12)))
armed_mage = Mage("Supermerlin", hp_max=60, damage=10,
                  weapon=Weapon("Nebula Staff", Dice(8)))
Battle(armed_warrior, armed_mage).simulate_battle()
assert armed_warrior.is_alive != armed_mage.is_alive
if armed_mage.is_alive:
    print("Remaining mana:", armed_mage.mana)

Neo with a gun HP: 45, Supermerlin HP: 54
Neo with a gun HP: 30, Supermerlin HP: 46
Neo with a gun HP: 15, Supermerlin HP: 45
Neo with a gun HP: 15, Supermerlin HP: 44
Neo with a gun HP: 15, Supermerlin HP: 37
Neo with a gun HP: 0, Supermerlin HP: 36
Neo with a gun has been defeated by Supermerlin. BUMMER!
Remaining mana: 0


## Results and limitations

The printed rounds show the impact of armor, mana consumption and random weapon damage. Compare the surviving character and remaining HP in the two battles. These are illustrative examples rather than evidence that one class is always stronger: the hero attacks first, dice vary and different health or damage settings can change the winner. Health always remains within bounds for valid initial maxima. The round limit stops configurations where neither character can win; reaching it does not declare a winner.

In [11]:
blocked_hero = Warrior("Sumowrestler", damage=5)
blocked_enemy = Warrior("Weightlifter", damage=5)
try:
    Battle(blocked_hero, blocked_enemy).simulate_battle(max_rounds=3)
except RuntimeError as error:
    print(error)
else:
    raise AssertionError("Blocked battle should reach its round limit")
assert blocked_hero.hp == blocked_enemy.hp == 100

fallen = Character("Fallen", hp_max=0)
survivor = Character("Survivor")
Battle(fallen, survivor).play_round()
assert survivor.hp == 100
print("Round limit and finished-battle checks passed.")

Sumowrestler HP: 100, Weightlifter HP: 100
Sumowrestler HP: 100, Weightlifter HP: 100
Sumowrestler HP: 100, Weightlifter HP: 100
The battle has lasted eons! Return home champions.
Round limit and finished-battle checks passed.


## GenAI Use – ChatGPT Conversation Summary

From chatGPT:

During Deliverable 3, I used ChatGPT as a learning assistant to help understand the assignment requirements, develop the RPG battle simulator, troubleshoot errors, and review my code.

Initially, ChatGPT helped me plan the project structure by recommending separate Python modules for the different classes and a Jupyter Notebook for documentation, testing, and battle simulations. It also provided guidance on creating and activating a virtual environment (`venv_d3`).

Throughout the development process, I wrote and shared my Python code with ChatGPT for feedback. The assistance mainly involved explaining object-oriented programming concepts such as encapsulation, inheritance, method overriding, composition, and the use of `super()`.

For the `Character` class, ChatGPT helped identify indentation errors, improve HP handling, and explain properties and encapsulation. For the `Dice` and `Weapon` classes, it helped identify mistakes involving method parameters, dice validation, and string representations.

When implementing `Warrior` and `Mage`, ChatGPT explained how subclasses inherit functionality from `Character` and how to override methods. It also helped identify issues with mana regeneration and damage being applied incorrectly.

For the `Battle` class, ChatGPT recommended using the characters' existing `attack()` methods rather than calculating damage directly, ensuring that weapons and special abilities worked correctly during combat.

ChatGPT also assisted with writing tests using `assert`, troubleshooting Jupyter import errors, and organizing the notebook to meet the assignment requirements. Towards the end, I uploaded the project files for review, and ChatGPT provided feedback on missing documentation, unnecessary code, and submission requirements.

**Overall, ChatGPT was used for explanations, debugging, code review, testing suggestions, and project organization.** The interaction was primarily iterative: I developed and shared code, received feedback, and made improvements based on the explanations. ChatGPT also provided example code snippets and test cases that helped guide the implementation.

Disclosing Codex use:

Codex was also used in the final stage to look at what changes could be made to fit the requierements in a more organized manner than the messy notebook i was working with. Codex ended up making more changes than i suggested so i spent some time putting back in the original solutions and rewriting the README.